In [1]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException

# from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd
from time import sleep

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)

driver.implicitly_wait(10) # 隱性等待

driver.get("https://today.line.me/tw/v2/movie/incinemas/playing")

# 初始化 ActionChains
action = ActionChains(driver)


In [ ]:
driver.quit()

### 利用 While Loop 與判斷式實作自動化網頁無限滾動

本小節將展示如何透過無限迴圈 (`while True`) 搭配條件判斷式 (`if`)，來應對現代網頁常見的**懶加載 (Lazy Loading)** 機制。透過模擬人類向下滾動網頁的行為，我們可以持續觸發 AJAX 請求，直到抓取完整頁面資料。

**核心執行步驟：**
1. **狀態追蹤**：設定 `last_count` 變數，用於紀錄前一次滾動後的資料總筆數。
2. **終止條件判斷**：在迴圈內抓取當前頁面的元素總數 (`current_count`)。若兩者數量相等 (`current_count == last_count`)，即代表網頁已抵達底部，無新資料載入，此時執行 `break` 終止迴圈。
3. **動態滾動觸發**：若尚有新資料，則定位出當前清單中的「最後一個元素」，並運用 `ActionChains` 將視窗滾動至該元素位置，迫使網頁載入後續資料。

In [4]:
last_height = driver.execute_script(" return document.body.scrollHeight ")
 
while True :
        # 2. 捲動到底部
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
       
        # 3. 等待頁面載入
        sleep(3)
       
        # 4. 計算新的頁面高度
        new_height = driver.execute_script("return document.body.scrollHeight")
       
        # 5. 比對高度是否改變
        if new_height == last_height:
            print("已抵達頁面底部")
            break
       
        # 更新高度基準
        last_height = new_height

movie_list_whole = driver.find_elements(By.CSS_SELECTOR, 'div.detailRankingItem.css-er84rx')

已抵達頁面底部


### 利用enumerate 將以取得的電影資訊篩選出我們要的內容存入清單內


#### 一、 核心教學目標：利用 `enumerate` 篩選並儲存資料
在處理多筆資料時，**「利用 `enumerate` 將已取得的電影資訊篩選出我們要的內容存入清單內」** 是一個非常標準且優雅的設計模式。`enumerate` 不僅能讓我們遍歷元素集合（`movie_list_whole`），還能同時自動生成索引值（`i`），這對於追蹤爬取進度、記錄日誌（Log）或在發生例外錯誤時進行除錯都極具價值。

#### 二、 Selenium 關鍵取值方法
在上述的迴圈中，我們主要練習兩種最常用的 Selenium 資料萃取屬性與方法：

1.  **`.text` 屬性**：
    * **用途**：取得兩個 HTML 標籤之間包含的純文字內容。
    * **應用場景**：當我們需要標題、評分、分級等直接顯示在網頁上的可見文字時使用。例如從 `<h2>標題</h2>` 中提取「標題」。
2.  **`.get_attribute('屬性名稱')` 方法**：
    * **用途**：取得 HTML 標籤內的特定屬性值。
    * **應用場景**：網頁中有許多重要資訊不會直接顯示為文字，而是隱藏在標籤屬性中。例如超連結的網址（`href`）、圖片的來源（`src`），或是表單元素的名稱（`name`）。在此範例中，我們利用 `.get_attribute('href')` 來精準提取時刻表的目標網址。


In [7]:
# 建立一個空的容器，放置電影資訊
movie_list = []

# 遍歷所有電影元素並提取詳細的電影資訊
for i, movie in enumerate(movie_list_whole):
    try:
        # 取得電影標題
        '''
        CSS Selector:
        h2.rankingItemHeader-title.header.css-8krq2c
        '''
        title = movie.find_element(By.CSS_SELECTOR, 'h2.rankingItemHeader-title.header.css-8krq2c').text
        
        # 取得評分
        '''
        CSS Selector :
        span.iconInfo-text.css-tue6l9
        '''
        score = movie.find_element(By.CSS_SELECTOR, 'span.iconInfo-text.css-tue6l9').text

        
        # 取得分級 
        '''
        CSS Selector :
        css-y8as7l
        '''
        # 初始化分級元素
        rating = '無分級資訊'
        try:
            rating = movie.find_element(By.CSS_SELECTOR, '.css-y8as7l').text
        # 若找不到元素，程式不會報錯，而是執行此區塊
        # 可以選擇裡面填寫 pass 或紀錄log
        except NoSuchElementException:
            pass

        # 取得時刻表連結
        #利用 .get_attribute('href') 來精準提取時刻表的目標網址
        '''
        CSS Selector:
        div.rankingItemAction.css-1n27rwt>a.ltcp-link
        '''
        showtime_link = movie.find_element(By.CSS_SELECTOR,'div.rankingItemAction.css-1n27rwt>a.ltcp-link').get_attribute('href')


    # 將單一電影的所有屬性打包為字典 (Dictionary)，並附加至 movie_list 串列的最尾端
    # 建立鍵值對 (Key-Value Pair) 以對應萃取出的變數
        print(f"{i+1}, {title}, 評分:{score}, 分級:{rating}, 場次連結:{showtime_link}")
        movie_list.append(
        {
            'title':title,
            'score':score,
            'rating':rating,
            'showtime_link': showtime_link,
        })

    except Exception as e :
        print(e) 

1, 另一個媽咪, 評分:5.8, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/mWR9WLn/1
2, 惡靈古堡：爆發夜, 評分:7.8, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/9mGGJjg/1
3, 不存在的罪人, 評分:8.0, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/aGy70l2/1
4, 野獸之心, 評分:9.1, 分級:保護級, 場次連結:https://today.line.me/tw/v3/movie/peKjGOw/1
5, 挖掘者, 評分:8.5, 分級:保護級, 場次連結:https://today.line.me/tw/v3/movie/yzWYDaz/1
6, 角落小夥伴電影版：從天而降的雲之冒險, 評分:10, 分級:普遍級, 場次連結:https://today.line.me/tw/v3/movie/2DBBkpO/1
7, 金劫任務, 評分:8.9, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/YapaywL/1
8, 復仇者聯盟：終局之戰(加碼重映版), 評分:8.6, 分級:保護級, 場次連結:https://today.line.me/tw/v3/movie/aGRQaLo/1
9, 蜘蛛人：重生日, 評分:9.1, 分級:保護級, 場次連結:https://today.line.me/tw/v3/movie/j7v6023/1
10, 劇場版 吉伊卡哇 人魚島的秘密, 評分:9.3, 分級:保護級, 場次連結:https://today.line.me/tw/v3/movie/wJ3xgvw/1
11, 愛你致死不渝, 評分:7.2, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/vXamQE8/1
12, 網紅老爸, 評分:8.5, 分級:輔15級, 場次連結:https://today.line.me/tw/v3/movie/nXKKeVL/1
13, 驚天劫機, 評分:9.7, 分級:限制級, 場次連結:https://today.line.me/tw

In [ ]:
'''
等等會用到
'''
FULL_STAR_PATH = 'M12 18.344l-5.81 3.609c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17.077.032.138.094.17.17l2.684 6.514 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343z'

HALF_STAR_PATH = 'M12.12 2.024c.076.031.137.093.169.17l2.684 6.513 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343l-5.81 3.61c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17zM12 6.463v9.651l3.662 2.275-.925-4.383 3.316-3.086-4.398-.443L12 6.463z'

from selenium.common.exceptions import TimeoutException, ElementNotInteractableException, NoSuchElementException, StaleElementReferenceException       

# 初始化一個主列表來存放評論資料
all_reviews_data = []

# 取的單一電影的場次連結 Showtime_link
for movie in movie_list:
    # 取得電影名稱
    Showtime_link = movie['showtime_link']
    driver.get(Showtime_link)

    movie_name = movie['title']
    try:
        '''
        一、挑戰(ㄧ): 訪問詳細頁面時，等待最多 10 秒，直到「網友短評」這個分頁元素可以點擊
            教學目標:
                - 設定並練習使用顯式等待 (Explicit Wait)的策略
            學員實作:
                - 建立一個等待物件，設定最長的等待時間為 10 秒
                - 後續所有的等待操作都須遵守這個超時設定
        '''
        wait = WebDriverWait(driver, 10)
        '''
        二、挑戰(二): 點擊「網友短評」這個分頁元素
           教學目標:
                - 當有多個元素共用同一個 CSS選擇器字串，要等到這些元素均出現，且「網友短評」這個分頁元素可以點擊
            學員實作:
                - 將 CSS 選擇器字串存放到一個變數中，方便後續維護與修改
                註 : CSS_Selector :  'a.ltcp-link.tabLink.movie.css-1abnfxj'
                - 撰寫邏輯使程式等待直到所有符合選擇器的分頁連結元素都出現在頁面中      
        '''

        # Selenium 的 expected_condition 模組中，所有與尋找元素相關的條件類別，其均設計為接收 「一個」參數
        # 而該參數被嚴格定義為一個包含兩個字串原物的元組(Tuple)
        # 因此當我們利用By策略時，傳入presence_of_all_elements_located這個方法的參數須為Tuple(By.CSS_SELECTOR, tab_selector)
        review_css_selector = 'a.ltcp-link.tabLink.movie.css-1abnfxj'

        elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, review_css_selector)))
        
        # 檢查分頁元素的數量
        if len(elements) < 3:
            print(f"電影{movie_name}的分頁數量不足，故跳過...")
            continue
        
        '''
        「如果你問老師說 : 那如果 10 秒內都沒有等到呢?」
        :「那就會拋出 TimeoutException 異常，直行將跳轉到對應的 except 區塊」
        '''

        # 取得「網友短評」分頁元素
        review_tab = elements[-1]

        # 等到元素可以被點擊，後進行點擊 (顯示等待)
        wait.until(EC.element_to_be_clickable(review_tab)).click()        

    # 捕捉在尋找或點擊「網友短評」元素過程中，最常發生的三種特定例外狀況：
    # 1. TimeoutException: 顯式等待（Explicit Wait）超過設定時間上限，目標元素仍未載入或不可見。
    # 2. IndexError: 嘗試透過索引值取出列表元素（例如 elements[0]）時，發現列表為空，意味著 DOM 樹中不存在符合選擇器的節點。
    # 3. ElementNotInteractableException: 目標元素雖然存在於 DOM 樹中，但可能被其他浮動元素（如廣告、彈出視窗）遮擋，或屬性為隱藏（Hidden），導致 WebDriver 無法執行點擊動作。
    except (TimeoutException, IndexError, ElementNotInteractableException) as e:
        print(f"{movie_name}電影分頁有誤，原因:{e}，故跳過")
    
    # # 恭喜，如果你在這邊還活著的話，你已經可以順路進入到網友短評
    # # 觀察每個「網友短評」，會發現依然要向下滾動網頁使瀏覽器載入新的評論

    last_height = driver.execute_script(" return document.body.scrollHeight ")
	
    while True :
		# 2. 捲動到底部
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		
        # 3. 等待頁面載入
        sleep(3)
        
        # 4. 計算新的頁面高度
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        # 5. 比對高度是否改變
        if new_height == last_height:
            print("已抵達頁面底部")
            break
        
        # 更新高度基準
        last_height = new_height

        comment_list_whole = driver.find_elements(By.CSS_SELECTOR,'div.css-hnvcda')

    
    # 遍歷當前頁面的所有評論
    for html in comment_list_whole:
        try:
            # 擷取星星數量
            # CSS_Selector : div.ratingStars.css-4rdtjp path
            star_html = html.find_elements(By.CSS_SELECTOR, 'div.ratingStars.css-4rdtjp path')

            # 初始化星星數量
            star = 0

            # 判定星星為整顆(FULL_STAR_PATH)/半顆(HALF_STAR_PATH)，並且計算星星數量
            for star_d in star_html:
                # 取得 path 的 d 屬性值
                star_path = star_d.get_attribute('d')

                if star_path == FULL_STAR_PATH:
                    star += 1
                elif star_path == HALF_STAR_PATH:
                    star += 0.5

            # 擷取評論的內容文字賦值給comment_text
            # CSS_Selector : span.css-12gpsi
            comment = html.find_element(By.CSS_SELECTOR, 'span.css-12gpsi').text

            #將擷取到的評論資料整理成字典，存入總清單中
            all_reviews_data.append({
                'movie_name': movie_name,
                'star': star,
                'comment': comment
            })
            
        except (NoSuchElementException,TimeoutException, ElementNotInteractableException, NoSuchElementException, StaleElementReferenceException) as e:
            print(f"電影{movie_name}的某則評論不完整或元素已失效，原因:{e}，故跳過")
            continue

# 將先前爬蟲迴圈中收集到包含多個字典(Dictionary)的串列(List) 'all_reviews_data'
# 作為引數傳入 pd.DataFrame() 建構子中。
# 此操作會將該串列轉換為二維的 Pandas DataFrame 物件，並將記憶體參考賦值給變數 'df'
df = pd.DataFrame(all_reviews_data)

driver.quit()

print(f"DataFrame Head:{df.head()}")
print(f"DataFrame info:{df.info()}")

已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
已抵達頁面底部
電影劇場版 鋼之鍊金術師 嘆息之丘的聖星的某則評論不完整或元素已失效，原因:Message: stale element reference: stale element not found
  (Session info: chrome=155.0.8059.79); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff6895f5805]
	(No symbol) [0x7ff689517d50]
	(No symbol) [0x7ff6893346dd]
	(No symbol) [0x7ff68934bffd]
	(No symbol) [0x7ff68934aa60]
	(No symbol) [0x7ff68933d425]
	(No symbol) [0x7ff68933b215]
	(No symbol) [0x7ff68933f6f5]
	(No symbol) [0x7ff68933f7bf]
	(No symbol) [0x7ff68938f82b]
	(No symbol) [0x7ff6893902bc]
	(No symbol) [0x7ff689383a4c]
	(No symbol) [0x7ff6893833e6]
	(No symbol) [0x7ff6893e54e1]
	(No symbol) [0x7ff689381d52]
	(No symbol) [0x7ff6893

存成 csv 檔案

In [12]:
# 檢視資料表前 5 列 (Rows)，確認資料是否正確載入且行名 (Columns) 對齊無誤
print("資料表前 5 列:")
print(df.head())

資料表前 5 列:
  movie_name  star                                            comment
0      另一個媽咪   1.0  傻眼第1次被預告騙進影城看\n預告明明就有一幕另一個媽媽和貝拉一起洗澡\n我一直在等這一幕結...
1      另一個媽咪   3.5  整體上算是好看，\n我覺得\n最有智慧的是貝拉的外婆，\n最有愛的是貝拉的爸比，\n最渣的是...
2      另一個媽咪   5.0  我覺得很好ㄚ\n要認真看導演想表達的寓意\n大家不要在婚姻中做一些有的沒 別人 自己小孩都無...
3      另一個媽咪   1.5                                    預告騙人，內容很無言，結局很瞎
4      另一個媽咪   2.0                               故事非常牽強，預告做的不錯被騙進戲院..


In [13]:
# 3. 檢視數值型資料行的敘述性統計 (如：評分的最大值、最小值、平均值)
print("\n數值資料敘述性統計:")
print(df.describe())


數值資料敘述性統計:
              star
count  1630.000000
mean      4.126687
std       1.308022
min       0.500000
25%       4.000000
50%       4.500000
75%       5.000000
max       5.000000


**`encoding='utf-8-sig'`**

：
**字元編碼參數**

。這是針對繁體中文語系至關重要的設定。一般寫入文字檔常使用

```
'utf-8'
```

，但在微軟的 Excel 軟體中直接開啟普通的 UTF-8 檔案時，經常會出現中文亂碼。加上

```
-sig
```

後，Pandas 會在檔案開頭寫入

**BOM（Byte Order Mark，位元組順序記號）**

，藉此強制作業系統與 Excel 識別該檔案為 UTF-8 編碼，確保中文字元得以正確渲染。

In [14]:
#  將清理或確認無誤的資料匯出為 CSV 檔案進行持久化儲存
# 參數說明：index=False 避免將 DataFrame 預設的整數索引寫入檔案；encoding='utf-8-sig' 確保繁體中文在 Excel 中不會出現亂碼
df.to_csv('movie_reviews.csv', index=False, encoding='utf-8-sig')